In [1]:
import numpy as np
import pandas as pd
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import datetime


In [2]:
def readActivities(path, subjectNumber):
    activity_table = pd.read_csv(path + "oura_daily_activity_data_cleaned_S" + str(subjectNumber) + ".csv")
    activity_table = activity_table[['day',
                                     'active_calories',
                                     'average_met_minutes',
                                     'equivalent_walking_distance',
                                     'high_activity_met_minutes',
                                     'high_activity_time',
                                     'steps',
                                     'total_calories']]
    return activity_table

In [3]:
def makeValidList(string):
    if (string == "[]"):
        return []
    temp = string[1:-1]
    temp = temp.split(sep = ", ")
    return [float(x) for x in temp if x != "None"]

def countNone(string):
    res = 0
    if (string == "[]"):
        return res
    temp = string[1:-1]
    temp = temp.split(sep = ", ")
    for x in temp:
        if x == "None":
            res += 1
    return res


def readSleepData(path, subjectNumber):
    sleep_table = pd.read_csv(path + "oura_sleep_data_cleaned_S" + str(subjectNumber) + ".csv")
    sleep_table = sleep_table[['day',
                               'heart_rate',
                               'hrv',
                               'bedtime_start',
                               'bedtime_end',
                               'deep_sleep_duration',
                               'light_sleep_duration',
                               'rem_sleep_duration',
                               'restless_periods',
                               'time_in_bed',
                               'total_sleep_duration']]
    sleep_table['HRVNoneCount'] = sleep_table['hrv'].apply(lambda x : countNone(x))
    sleep_table['hrv'] = sleep_table['hrv'].apply(lambda x: makeValidList(x))
    sleep_table['hrvCount'] = sleep_table['hrv'].apply(lambda x: len(x))
    sleep_table['HRNoneCount'] = sleep_table['heart_rate'].apply(lambda x : countNone(x))
    sleep_table['heart_rate'] = sleep_table['heart_rate'].apply(lambda x: makeValidList(x))
    sleep_table['heart_rateCount'] = sleep_table['heart_rate'].apply(lambda x: len(x))
    return sleep_table



In [4]:
def aggregateSleepDataByDay(sleepTable,subjectNumber):
    aggregate_sleep_statistics = ['hrv',
                                  'HRVNoneCount',
                                  'heart_rate',
                                  'HRNoneCount',
                                  'hrvCount',
                                  'heart_rateCount',
                                  'deep_sleep_duration',
                                  'light_sleep_duration',
                                  'rem_sleep_duration',
                                  'restless_periods',
                                  'time_in_bed',
                                  'total_sleep_duration']
    data = {'day' : list(sleepTable['day'].unique())}
    A = pd.DataFrame(data = data)
    l = len(A)
    for aggstat in aggregate_sleep_statistics:
        newcol = []
        for i in range(l):
            tempdf = sleepTable[(sleepTable['day'] == A.iloc[i]['day'])]
            if (aggstat == "hrv") or (aggstat == "heart_rate"):
                temp = []
            else: 
                temp = 0
            for j in range(len(tempdf)):
                temp = temp + tempdf.iloc[j][aggstat]
            newcol.append(temp)
        A.insert(loc = 0, column = aggstat, value = newcol)
    newcol = []
    newcol2 = []
    for i in range(l):
        tempdf = sleepTable[(sleepTable['day'] == A.iloc[i]['day'])]
        temp2 = 0
        for j in range(len(tempdf)):
            temp2 += 1
        newcol.append(temp)
        newcol2.append(temp2)
    A.insert(loc = 0, column = 'registeredSleepCount', value = newcol2)
    return A

In [5]:
def check_valid(string, seen):
    dt = datetime.datetime.fromisoformat(string)
    date = dt.date()
    if (dt.hour) < 7:
        return (False, seen)
    if date in seen:
        return (False, seen)
    seen.append(date)
    return (True, seen)



def readCognitiveTestData(path, subjectNumber):
    if subjectNumber < 10:
        response_table = pd.read_csv(path + "daily_stats_S0" + str(subjectNumber) + ".csv")
    else:
        response_table = pd.read_csv(path + "daily_stats_S" + str(subjectNumber) + ".csv")
    response_table = response_table[['Date and Time',
                                     'Correct Go Trials',
                                     'Avg RT Correct Go (ms)',
                                     'Correct No-Go Trials',
                                     'Avg RT Correct No-Go (ms)']]
    response_table['Date and Time'] = response_table['Date and Time'].apply(lambda x: x + '-05:00')
    temp = response_table['Date and Time'].copy()
    ind = list(temp.index)
    seen = []
    for i in ind:
        bl, seen = check_valid(response_table.loc[i,"Date and Time"], seen)
        response_table.loc[i,'valid'] = bl
    response_table = response_table[response_table['valid'] == True]
    response_table = response_table.drop(labels = 'valid', axis = 1)
    response_table['day'] = response_table['Date and Time'].apply(lambda x: datetime.datetime.fromisoformat(x).date().isoformat())
    if subjectNumber == 1:
        response_table.loc[:,'Correct Go Trials'] = response_table.loc[:,'Correct Go Trials'] / 20
        response_table.loc[:,'Correct No-Go Trials'] = response_table.loc[:,'Correct No-Go Trials'] / 10
    else:
        response_table.loc[:,'Correct Go Trials'] = response_table.loc[:,'Correct Go Trials'] / 35
        response_table.loc[:,'Correct No-Go Trials'] = response_table.loc[:,'Correct No-Go Trials'] / 15
    return response_table




In [6]:
def dataCleanBySubject(path, subjectNumber):
    A = readSleepData(path,subjectNumber)
    A = aggregateSleepDataByDay(A,subjectNumber)
    B = readCognitiveTestData(path,subjectNumber)
    C = readActivities(path, subjectNumber)
    return (A,B,C)

In [7]:
def calculateStatistics(initial, features, data):
    res = initial.copy()
    for feature in features:
        result = []
        for df in data:
            temp = df.copy()[feature]
            mean = temp.mean()
            temp["Deviations"] = temp - mean
            devs = temp["Deviations"].copy()
            std = math.sqrt(devs.apply(lambda x: x**2).mean())
            result.append({
                'subject_no': df["subject_no"][0],
                feature + " (StDev)" : std,
                feature + " (CV)" : std/mean
            })
        res = pd.merge(res,pd.DataFrame(result), on = "subject_no")
    return res



def calculateSleepFeatures(initial, data):
    result = []
    for df in data:
        temp = df.copy()
        HRS = np.array(temp['heart_rate'].sum())
        HRVS = np.array(temp['hrv'].sum())
        LnHRS = np.array([math.log(hr) for hr in HRS])
        LnHRVS = np.array([math.log(hrv) for hrv in HRVS])
        medianHR = np.median(HRS)
        medianHRV = np.median(HRVS)
        avgHR = np.mean(HRS)
        avgHRV = np.mean(HRVS)
        avgLnHR = np.mean(LnHRS)
        avgLnHRV = np.mean(LnHRVS)
        stdHR = np.std(HRS)
        stdHRV = np.std(HRVS)
        stdLnHR = np.std(LnHRS)
        stdLnHRV = np.std(LnHRVS)
        efficiency = (temp['total_sleep_duration'].sum())/(temp['time_in_bed'].sum())
        HRNoneCount = temp["HRNoneCount"].sum()
        HRVNoneCount = temp["HRVNoneCount"].sum()
        sleepCountPerDay = np.mean(np.array(temp['registeredSleepCount']))
        result.append({
            'subject_no': df["subject_no"][0],
            'medianHR' : medianHR,
            'medianHRV' : medianHRV,
            'avgHR' : avgHR,
            'avgHRV' : avgHRV,
            'stdHR' : stdHR,
            'stdHRV' : stdHRV,
            'avgLnHR' : avgLnHR,
            'avgLnHRV' : avgLnHRV,
            'stdLnHR' : stdLnHR,
            'stdLnHRV' : stdLnHRV,
            'efficiency' : efficiency,
            'HRNoneCount' : HRNoneCount,
            'HRVNoneCount' : HRVNoneCount,
            'sleepCountPerDay': sleepCountPerDay
        })
    res = pd.merge(initial.copy(),pd.DataFrame(result), on = "subject_no")
    return res


cogs = ['Correct Go Trials', 'Avg RT Correct Go (ms)', 'Correct No-Go Trials', 'Avg RT Correct No-Go (ms)']



aggregateSleepFeatures = [
                 "total_sleep_duration",
                 "time_in_bed",
                 "restless_periods",
                 "rem_sleep_duration",
                 "light_sleep_duration",
                 "deep_sleep_duration"]

activityFeatures = ['active_calories',
                    'average_met_minutes',
                    'equivalent_walking_distance',
                    'high_activity_met_minutes',
                    'high_activity_time',
                    'steps',
                    'total_calories']


In [8]:
def dataClean(path, numSubjects, BRS):
    initial = []
    sleepData = []
    cogData = []
    activityData = []    
    for i in range(numSubjects):
        if ((i != 0) & (i != 1) & (i != 5)):
            score = BRS[str(i+1)]
            subjectSleepData, subjectCogData, subjectActivityData = dataCleanBySubject(path, i+1)
            initial.append({
                'subject_no' : i+1,
                'avg_brs' : score
            })
            subjectSleepData.insert(loc = 0, column = 'subject_no', value = [i+1 for a in range(len(subjectSleepData))])
            sleepData.append(subjectSleepData)
            subjectCogData.insert(loc = 0, column = 'subject_no', value = [i+1 for a in range(len(subjectCogData))])
            cogData.append(subjectCogData)
            subjectActivityData.insert(loc = 0, column = 'subject_no', value = [i+1 for a in range(len(subjectActivityData))])
            activityData.append(subjectActivityData)
    initial = pd.DataFrame(initial)
    return (initial, sleepData, cogData, activityData)

In [9]:
def generateDataByDay(initial, sleepData, cogData, activityData):
    res = []
    for i in range(len(initial)):
        sleep = sleepData[i]
        cog = cogData[i]
        activity = activityData[i]
        sleep = sleep.drop('subject_no', axis = 1)
        cog = cog.drop('subject_no', axis = 1)
        activity = activity.drop('subject_no', axis = 1)
        temp = pd.merge(sleep, cog, how = "inner", on = "day")
        temp = pd.merge(temp, activity, how = "inner", on = "day")
        l = len(temp)
        score = initial.iloc[i,1]
        temp.insert(loc = 0, column = 'avg_brs', value = [score for i in range(l)])
        res.append(temp) 
    table = pd.concat(res)
    table = table[table['heart_rateCount'] > 0]
    table = table.drop(['day', 'registeredSleepCount','heart_rateCount','hrvCount','HRNoneCount','HRVNoneCount','day','Date and Time'], axis = 1)
    table['heart_rate'] = table['heart_rate'].apply(lambda x: np.mean(np.array(x)))
    table['hrv'] = table['hrv'].apply(lambda x: np.mean(np.array(x)))
    y = table['avg_brs']
    X = table.drop('avg_brs',axis = 1)
    scaler = StandardScaler(with_mean = True, with_std = True, copy = True)
    scaler.fit(X)
    X_std = scaler.transform(X)
    feature_std = pd.DataFrame(X_std,columns = X.columns)
    X_train, X_test, y_train, y_test = train_test_split(feature_std,y,test_size = 0.2, stratify = y, random_state = 243)
    X_train.to_csv("DataByDayXTrain.csv", index=False)
    X_test.to_csv("DataByDayXTest.csv", index=False)
    y_train.to_csv("DataByDayYTrain.csv", index=False)
    y_test.to_csv("DataByDayYTest.csv", index=False)

In [10]:
BRS = { "1" : 4, "2" : 19/6, "3" : 10/3, "4" : 10/3, "5" : 10/3, "6" : 22/6, "7" : 22/6, "8" : 26/6, "9" : 20/6, "10" : 4 , "11" : 23/6} 

initial, sleepData, cogData, activityData = dataClean("",11,BRS)

generateDataByDay(initial, sleepData, cogData, activityData)

In [11]:
table = calculateStatistics(initial, cogs, cogData)
table = calculateStatistics(table, aggregateSleepFeatures, sleepData)
table = calculateStatistics(table, activityFeatures, activityData)
table = calculateSleepFeatures(table, sleepData)
                       
table.to_csv("DataBySubject.csv", index=False)
print(table.info())
print(table.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 50 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   subject_no                           8 non-null      int64  
 1   avg_brs                              8 non-null      float64
 2   Correct Go Trials (StDev)            8 non-null      float64
 3   Correct Go Trials (CV)               8 non-null      float64
 4   Avg RT Correct Go (ms) (StDev)       8 non-null      float64
 5   Avg RT Correct Go (ms) (CV)          8 non-null      float64
 6   Correct No-Go Trials (StDev)         8 non-null      float64
 7   Correct No-Go Trials (CV)            8 non-null      float64
 8   Avg RT Correct No-Go (ms) (StDev)    8 non-null      float64
 9   Avg RT Correct No-Go (ms) (CV)       8 non-null      float64
 10  total_sleep_duration (StDev)         8 non-null      float64
 11  total_sleep_duration (CV)           